# 3D Computer Vision Project — Demo

End-to-end walkthrough of the pipeline:
1. Checkerboard corner detection
2. Camera calibration (Zhang's method) — intrinsics `K` + per-view extrinsics `(rvec, tvec)`
3. Object detection (cubes, targets, robot) in a scene image
4. Top-down 2D world view of detected objects

## Initialization

In [ ]:
import os, glob
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

from checkerboard_detection import _find_checkerboard_corners
from camera_calibration import calibrate, _build_object_points, _project_points
from object_detection import _detect_cubes, _detect_target_locations, _detect_robot, CUBE_TOP_HEIGHT_CM

CALIBRATION_DIR = 'test-images/set2/calibration'
SCENE_DIR = 'test-images/set2/images'

# Checkerboard details
PATTERN_SIZE = (8, 6)
SQUARE_SIZE_CM = 4.0

## 1. Checkerboard detection

Detect the 8x6 interior-corner grid in each calibration image using OpenCV's SB detector, with sub-pixel refinement.

In [ ]:
calib_paths  = sorted(glob.glob(os.path.join(CALIBRATION_DIR, '*.png')))
calib_images = [Image.open(p) for p in calib_paths]
print(f'Loaded {len(calib_images)} calibration image(s) from {CALIBRATION_DIR}')

for i, image in enumerate(calib_images):
    gray = np.array(image.convert('L'), dtype=np.uint8)
    try:
        corners, _ = _find_checkerboard_corners(gray, PATTERN_SIZE)
    except RuntimeError:
        print(f'Corners not detected in image {i}')
        continue
    plt.figure()
    plt.imshow(image)
    plt.scatter(corners[:, 0], corners[:, 1], s=4, c='red', marker='.')
    plt.title(f'Image {i}  —  {len(corners)} corners detected')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 2. Camera calibration (Zhang's method)

Pipeline: DLT homography per view -> Zhang's closed-form K (with EXIF fallback if the closed-form solution is implausible) -> per-view extrinsics -> joint LM refinement.

In [ ]:
calibration = calibrate(calib_images, PATTERN_SIZE, SQUARE_SIZE_CM)

K      = calibration['K']
rvecs  = calibration['rvecs']
tvecs  = calibration['tvecs']
w, h   = calibration['image_size']

print('\n=== Calibration result ===')
print(f'Image size:    {w} x {h}')
print(f'Intrinsics K:\n{np.array2string(K, precision=2)}')
print(f'Focal length:  fx={K[0, 0]:.1f}, fy={K[1, 1]:.1f}')
print(f'Principal pt:  cx={K[0, 2]:.1f}, cy={K[1, 2]:.1f}')
print(f'Views used:    {len(rvecs)}')

### Reprojection + world frame on each view
Green circles = detected corners. Red x = reprojected (via refined `K, R, t`). Red/green/blue lines = world X/Y/Z axes at the board origin.

In [ ]:
object_points  = _build_object_points(PATTERN_SIZE, SQUARE_SIZE_CM)
axis_length_cm = 3 * SQUARE_SIZE_CM
world_axes = np.array([[0, 0, 0],
                       [axis_length_cm, 0, 0],
                       [0, axis_length_cm, 0],
                       [0, 0, axis_length_cm]], dtype=np.float64)

for i, pil_image in enumerate(calib_images):
    gray = np.array(pil_image.convert('L'), dtype=np.uint8)
    try:
        detected_corners, _ = _find_checkerboard_corners(gray, PATTERN_SIZE)
    except RuntimeError:
        continue

    projected_corners = _project_points(object_points.astype(np.float64),
                                        K, rvecs[i], tvecs[i])
    per_view_rms = np.sqrt(np.mean(
        np.sum((projected_corners - detected_corners) ** 2, axis=1)))
    axes_image = _project_points(world_axes, K, rvecs[i], tvecs[i])
    origin_px  = axes_image[0]

    plt.figure()
    plt.imshow(pil_image)
    plt.scatter(detected_corners[:, 0], detected_corners[:, 1], s=30,
                facecolors='none', edgecolors='green', linewidths=1.2, label='detected')
    plt.scatter(projected_corners[:, 0], projected_corners[:, 1], s=6,
                c='red', marker='x', label='reprojected')
    for axis_index, color, label in [(1, 'red', 'X'), (2, 'green', 'Y'), (3, 'blue', 'Z')]:
        end_px = axes_image[axis_index]
        plt.plot([origin_px[0], end_px[0]], [origin_px[1], end_px[1]],
                 color=color, linewidth=2.5)
        plt.text(end_px[0], end_px[1], label, color=color,
                 fontsize=12, fontweight='bold')
    plt.title(f'View {i}  -  per-view RMS: {per_view_rms:.2f} px')
    plt.legend(loc='upper right')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 3. Object detection in a scene image

Reuse the last calibration view's pose `(R, t)` as the scene pose (same camera position). Then run HSV-based blob detection for cubes (solid blobs, top face at z=4 cm), targets (rings with holes, on the floor), and the robot's yellow front / purple back markers.

In [ ]:
R_scene, _ = cv2.Rodrigues(calibration['rvecs'][-1])
t_scene    = calibration['tvecs'][-1].ravel()

scene_paths = sorted(glob.glob(os.path.join(SCENE_DIR, '*.png')))
scene_image = Image.open(scene_paths[0])
bgr = cv2.cvtColor(np.array(scene_image), cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

cubes   = _detect_cubes(hsv, calibration, R_scene, t_scene)
targets = _detect_target_locations(hsv, calibration, R_scene, t_scene)
try:
    robot = _detect_robot(hsv, calibration, R_scene, t_scene)
except RuntimeError as e:
    print(f'Robot detection failed: {e}')
    robot = None

print(f'\n=== Detection on {os.path.basename(scene_paths[0])} ===')
print(f'Cubes ({len(cubes)}):')
for color, xyz in cubes.items():
    print(f'  {color:6s}: ({xyz[0]:+7.1f}, {xyz[1]:+7.1f}, {xyz[2]:+5.1f}) cm')
print(f'Targets ({len(targets)}):')
for color, xyz in targets.items():
    print(f'  {color:6s}: ({xyz[0]:+7.1f}, {xyz[1]:+7.1f}, {xyz[2]:+5.1f}) cm')
if robot is not None:
    print(f"Robot pos: ({robot['pos'][0]:+7.1f}, {robot['pos'][1]:+7.1f}, "
          f"{robot['pos'][2]:+5.1f}) cm, heading: {np.degrees(robot['heading']):+6.1f} deg")

### Detections overlaid on the scene image
World points projected back to pixels using the scene pose. Cubes = filled squares; targets = open circles; robot = yellow front / magenta back markers with a line between them.

In [ ]:
def world_to_pixel(world_xyz):
    camera_pt = R_scene @ np.asarray(world_xyz) + t_scene
    image_pt  = K @ camera_pt
    return image_pt[0] / image_pt[2], image_pt[1] / image_pt[2]

plt.figure()
plt.imshow(scene_image)

origin_px = world_to_pixel([0.0, 0.0, 0.0])
for i, (axis_vec, color, label) in enumerate([
        ([SQUARE_SIZE_CM, 0, 0], 'red',   'X'),
        ([0, SQUARE_SIZE_CM, 0], 'green', 'Y'),
        ([0, 0, SQUARE_SIZE_CM], 'blue',  'Z')]):
    end_px = world_to_pixel(axis_vec)
    plt.plot([origin_px[0], end_px[0]], [origin_px[1], end_px[1]],
             color=color, linewidth=2.5, zorder=i+2)
    plt.text(end_px[0], end_px[1], label, color=color,
             fontsize=12, fontweight='bold', zorder=i+2)

# Cubes at their actual top-face height, plus a faint floor shadow at z=0
for color, xyz in cubes.items():
    u, v = world_to_pixel(xyz)
    us, vs = world_to_pixel([xyz[0], xyz[1], 0.0])
    plt.scatter(us, vs, s=140, facecolors='none', edgecolors=color,
                linewidths=1.0, linestyle=':', marker='s', alpha=0.7, zorder=1)
    plt.scatter(u, v, s=140, facecolors=color, edgecolors=color,
                linewidths=2.0, marker='s', label=f'cube:{color}')
for color, xyz in targets.items():
    u, v = world_to_pixel(xyz)
    plt.scatter(u, v, s=140, facecolors='none', edgecolors=color,
                linewidths=5.0, marker='o', label=f'target:{color}')
if robot is not None:
    uf, vf = world_to_pixel(robot['front'])
    ub, vb = world_to_pixel(robot['back'])
    # Floor shadow of the robot body
    ufs, vfs = world_to_pixel([robot['front'][0], robot['front'][1], 0.0])
    ubs, vbs = world_to_pixel([robot['back'][0],  robot['back'][1],  0.0])
    plt.plot([ubs, ufs], [vbs, vfs], color='gray', linewidth=1.0,
             linestyle=':', alpha=0.6, zorder=0)
    plt.scatter(ufs, vfs, s=60, facecolors='none', edgecolors='gray',
                linewidths=1.0, marker='o', alpha=0.6, zorder=0)
    plt.scatter(ubs, vbs, s=60, facecolors='none', edgecolors='gray',
                linewidths=1.0, marker='o', alpha=0.6, zorder=0)
    plt.plot([ub, uf], [vb, vf], color='yellow', linewidth=3, zorder=1)
    plt.scatter(uf, vf, s=120, c='yellow', marker='.', linewidths=2.5,
                zorder=1, label='robot front')
    plt.scatter(ub, vb, s=120, c='magenta', marker='.', linewidths=2.5,
                zorder=2, label='robot back')

plt.title(f"Detections on {os.path.basename(scene_paths[0])}")
plt.legend()
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Top-down world view
Plotted in real-world (X, Y) cm. Each object's `(X, Y)` is the metric position at its actual height plane: cubes at z=4, targets at z=0, robot markers at their physical heights.

In [ ]:
plt.figure()

plt.plot([0, SQUARE_SIZE_CM], [0, 0], color='red',   linewidth=2.5)
plt.plot([0, 0], [0, SQUARE_SIZE_CM], color='green', linewidth=2.5)
plt.text(SQUARE_SIZE_CM, 0, 'X', color='red',   fontsize=12, fontweight='bold')
plt.text(0, SQUARE_SIZE_CM, 'Y', color='green', fontsize=12, fontweight='bold')

for color, xyz in cubes.items():
    plt.scatter(xyz[0], xyz[1], s=140, facecolors=color, edgecolors=color,
                linewidths=2.0, marker='s', label=f'cube:{color}')
for color, xyz in targets.items():
    plt.scatter(xyz[0], xyz[1], s=140, facecolors='none', edgecolors=color,
                linewidths=5.0, marker='o', label=f'target:{color}')
if robot is not None:
    bx, by, _  = robot['back']
    fx_, fy_, _ = robot['front']
    plt.plot([bx, fx_], [by, fy_], color='yellow', linewidth=3, zorder=1)
    plt.scatter(fx_, fy_, s=120, c='yellow', marker='.', linewidths=2.5,
                zorder=1, label='robot front')
    plt.scatter(bx, by, s=120, c='magenta', marker='.', linewidths=2.5,
                zorder=2, label='robot back')

plt.xlabel('X (cm)')
plt.ylabel('Y (cm)')
plt.title('Top-down world view')
plt.axis('equal')
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Robot control — plan pick-and-place

For each cube colour, call `_plan_for_color` to get a structured list of plan steps. The planner mutates `robot_state` so consecutive calls chain naturally. `_translate` then serializes the steps to the assignment's `cmd; cmd; ...` string format.

In [ ]:
from robot_control import (
    _plan_for_color,
    _translate,
    GRABBER_OFFSET_CM,
)

BLOCK_ORDER = ["red", "green", "blue"]

robot_state = {
    "pos": robot["pos"][:2].copy(),
    "heading": robot["heading"],
}
all_steps = []

print(f"Robot start: ({robot_state['pos'][0]:+.1f}, {robot_state['pos'][1]:+.1f}) cm, "
      f"heading: {np.degrees(robot_state['heading']):+.1f} deg")

for color in BLOCK_ORDER:
    try:
        steps = _plan_for_color(color, robot_state, cubes, targets)
    except KeyError as e:
        print(f"Skipping {color}: {e}")
        continue
    print(f"\n-- {color} --")
    print(f"  commands: {_translate(steps)}")
    all_steps.extend(steps)

n_grabs = sum(1 for step in all_steps if step['cmd'] == 'grab')
print(f"\nEmitted {len(all_steps)} step(s) ({n_grabs} pick-and-place cycles).")
print(f"\nFinal command string:\n{_translate(all_steps)}")

## 6. Animate the robot

Simulate the robot's pose over time by densely sampling each plan step: turns sweep the heading, straight drives interpolate position, and `grab`/`let_go` toggle which cube (if any) is being carried. The first animation shows a top-down world view; the second overlays the same motion on the scene image using the calibrated pose.

In [ ]:
import matplotlib.animation as animation
import matplotlib as mpl
from IPython.display import HTML
from object_detection import ROBOT_FRONT_HEIGHT_CM

# Allow larger embedded animations (default 20 MB is too small for ~200 frames)
mpl.rcParams['animation.embed_limit'] = 64.0

# Which cube colour is moved in each pick-and-place cycle (in BLOCK_ORDER,
# skipping any colour whose cube/target wasn't detected)
moved_colors = [c for c in BLOCK_ORDER if c in cubes and c in targets]

def simulate(steps, start_pos, start_heading, cycle_colors,
             samples_per_turn_deg=2.0, samples_per_go=8):
    pos = np.asarray(start_pos, dtype=float).copy()
    heading = float(start_heading)
    cycle = 0
    held = None
    frames = [(pos.copy(), heading, held)]
    for step in steps:
        if step['cmd'] == 'turn':
            total_deg = step['param']
            n = max(2, int(abs(total_deg) / samples_per_turn_deg) + 1)
            for frac in np.linspace(0.0, 1.0, n)[1:]:
                h = heading + np.radians(total_deg) * frac
                frames.append((pos.copy(), h, held))
            heading = step['heading']
        elif step['cmd'] == 'go':
            end_pos = step['pos']
            for frac in np.linspace(0.0, 1.0, samples_per_go)[1:]:
                frames.append((pos + frac * (end_pos - pos), heading, held))
            pos = end_pos.copy()
        elif step['cmd'] == 'grab':
            held = cycle_colors[cycle] if cycle < len(cycle_colors) else None
            frames.append((pos.copy(), heading, held))
        elif step['cmd'] == 'let_go':
            held = None
            frames.append((pos.copy(), heading, held))
            cycle += 1
    return frames

start_pos = np.asarray(robot['pos'][:2], dtype=float)
start_heading = float(robot['heading'])
frames = simulate(all_steps, start_pos, start_heading, moved_colors)
print(f'Generated {len(frames)} animation frames.')

### Top-down animation

In [ ]:
fig_top, ax_top = plt.subplots(figsize=(10, 6))

for c, xyz in cubes.items():
    ax_top.scatter(xyz[0], xyz[1], s=140, c=c, marker='s', alpha=0.35,
                   edgecolors='none', label=f'cube start:{c}')
for c, xyz in targets.items():
    ax_top.scatter(xyz[0], xyz[1], s=140, facecolors='none', edgecolors=c,
                   linewidths=5.0, marker='o', label=f'target:{c}')

center_trail,  = ax_top.plot([], [], color='magenta', linestyle='--',
                             linewidth=1.2, zorder=1, label='center trail')
grabber_trail, = ax_top.plot([], [], color='yellow', linewidth=1.5,
                             zorder=2, label='grabber trail')
body_line,     = ax_top.plot([], [], color='yellow', linewidth=2.5, zorder=3)
center_dot     = ax_top.scatter([], [], s=100, c='magenta', marker='o',
                                edgecolors='none', zorder=4)
grabber_dot    = ax_top.scatter([], [], s=80, c='yellow', marker='o',
                                edgecolors='none', zorder=4)
held_dot       = ax_top.scatter([], [], s=160, marker='s',
                                edgecolors='none', zorder=5)
status_text    = ax_top.text(0.02, 0.98, '', transform=ax_top.transAxes,
                             va='top', ha='left', fontsize=10,
                             bbox=dict(facecolor='white', alpha=0.75, edgecolor='gray'))

all_points = np.vstack([np.array([p for (p, _, _) in frames]),
                        np.array([p + GRABBER_OFFSET_CM *
                                  np.array([np.cos(h), np.sin(h)])
                                  for (p, h, _) in frames])])
pad = 5.0
ax_top.set_xlim(all_points[:, 0].min() - pad, all_points[:, 0].max() + pad)
ax_top.set_ylim(all_points[:, 1].min() - pad, all_points[:, 1].max() + pad)
ax_top.set_xlabel('X (cm)')
ax_top.set_ylabel('Y (cm)')
ax_top.set_title('Robot pick-and-place animation (top-down)')
ax_top.set_aspect('equal')
ax_top.grid(True, linestyle=':', alpha=0.5)

def animate_top(i):
    past = frames[: i + 1]
    centers  = np.array([p for (p, _, _) in past])
    grabbers = np.array([p + GRABBER_OFFSET_CM * np.array([np.cos(h), np.sin(h)])
                         for (p, h, _) in past])
    pos, heading, held = frames[i]
    tip = pos + GRABBER_OFFSET_CM * np.array([np.cos(heading), np.sin(heading)])

    center_trail.set_data(centers[:, 0], centers[:, 1])
    grabber_trail.set_data(grabbers[:, 0], grabbers[:, 1])
    body_line.set_data([pos[0], tip[0]], [pos[1], tip[1]])
    center_dot.set_offsets([pos])
    grabber_dot.set_offsets([tip])
    if held is not None:
        held_dot.set_offsets([tip])
        held_dot.set_facecolor(held)
        held_dot.set_visible(True)
    else:
        held_dot.set_visible(False)
    status_text.set_text(
        f'frame {i+1}/{len(frames)}\n'
        f'pos: ({pos[0]:+.1f}, {pos[1]:+.1f}) cm\n'
        f'heading: {np.degrees(heading):+.1f} deg\n'
        f'holding: {held or "—"}')
    return (center_trail, grabber_trail, body_line, center_dot,
            grabber_dot, held_dot, status_text)

anim_top = animation.FuncAnimation(
    fig_top, animate_top, frames=len(frames),
    interval=50, blit=False, repeat=False)
plt.close(fig_top)  # Prevent duplicate static figure; show only the player
HTML(anim_top.to_jshtml())

### Scene-image animation

In [ ]:
def world_to_pixel_scene(world_xyz):
    camera_pt = R_scene @ np.asarray(world_xyz, dtype=float) + t_scene
    image_pt = calibration['K'] @ camera_pt
    return image_pt[0] / image_pt[2], image_pt[1] / image_pt[2]

# Precompute pixel positions for every frame (grabber tip at its height, plus
# the tip's floor shadow at z=0 so we can draw the held cube's projection)
frames_px = []
for (pos, heading, held) in frames:
    tip_world = np.array([pos[0] + GRABBER_OFFSET_CM * np.cos(heading),
                          pos[1] + GRABBER_OFFSET_CM * np.sin(heading),
                          ROBOT_FRONT_HEIGHT_CM])
    tip_floor = [tip_world[0], tip_world[1], 0.0]
    frames_px.append({
        'center':     world_to_pixel_scene([pos[0], pos[1], 0.0]),
        'tip':        world_to_pixel_scene(tip_world),
        'tip_shadow': world_to_pixel_scene(tip_floor),
        'held':       held,
    })

fig_img, ax_img = plt.subplots()
ax_img.imshow(scene_image)
# Cubes at their true top-face height plus a faint floor shadow
for c, xyz in cubes.items():
    u, v = world_to_pixel_scene(xyz)
    us, vs = world_to_pixel_scene([xyz[0], xyz[1], 0.0])
    ax_img.scatter(us, vs, s=140, facecolors='none', edgecolors=c,
                   linewidths=1.0, linestyle=':', marker='s', alpha=0.7, zorder=1)
    ax_img.scatter(u, v, s=140, c=c, marker='s', alpha=0.35,
                   edgecolors='none', zorder=2)
for c, xyz in targets.items():
    u, v = world_to_pixel_scene(xyz)
    ax_img.scatter(u, v, s=140, facecolors='none', edgecolors=c,
                   linewidths=5.0, marker='o')

center_trail_px,  = ax_img.plot([], [], color='magenta', linestyle='--',
                                linewidth=1.2, zorder=3)
grabber_trail_px, = ax_img.plot([], [], color='yellow', linewidth=1.5, zorder=4)
body_line_px,     = ax_img.plot([], [], color='yellow', linewidth=3, zorder=5)
center_dot_px     = ax_img.scatter([], [], s=80, c='magenta', marker='o',
                                   edgecolors='none', zorder=6)
grabber_dot_px    = ax_img.scatter([], [], s=60, c='yellow', marker='o',
                                   edgecolors='none', zorder=6)
held_dot_px       = ax_img.scatter([], [], s=120, marker='s',
                                   edgecolors='none', zorder=7)
held_shadow_px    = ax_img.scatter([], [], s=120, facecolors='none', marker='s',
                                   linewidths=1.0, linestyle=':', alpha=0.8,
                                   zorder=3)

def animate_img(i):
    centers_u = [frames_px[j]['center'][0] for j in range(i + 1)]
    centers_v = [frames_px[j]['center'][1] for j in range(i + 1)]
    tips_u    = [frames_px[j]['tip'][0]    for j in range(i + 1)]
    tips_v    = [frames_px[j]['tip'][1]    for j in range(i + 1)]
    f = frames_px[i]
    center_trail_px.set_data(centers_u, centers_v)
    grabber_trail_px.set_data(tips_u, tips_v)
    body_line_px.set_data([f['center'][0], f['tip'][0]],
                          [f['center'][1], f['tip'][1]])
    center_dot_px.set_offsets([f['center']])
    grabber_dot_px.set_offsets([f['tip']])
    if f['held'] is not None:
        held_dot_px.set_offsets([f['tip']])
        held_dot_px.set_facecolor(f['held'])
        held_dot_px.set_visible(True)
        held_shadow_px.set_offsets([f['tip_shadow']])
        held_shadow_px.set_edgecolor(f['held'])
        held_shadow_px.set_visible(True)
    else:
        held_dot_px.set_visible(False)
        held_shadow_px.set_visible(False)
    return (center_trail_px, grabber_trail_px, body_line_px,
            center_dot_px, grabber_dot_px, held_dot_px, held_shadow_px)

anim_img = animation.FuncAnimation(
    fig_img, animate_img, frames=len(frames),
    interval=50, blit=False, repeat=False)
ax_img.set_title(f"Robot animation on {os.path.basename(scene_paths[0])}")
ax_img.axis('off')
plt.close(fig_img)
HTML(anim_img.to_jshtml())